In [ ]:
import sys
import torch
from torch.utils.data import DataLoader
from torch import nn

import numpy as np

import torchvision.datasets as datasets
from torchvision.transforms import ToTensor

mnist_train = datasets.MNIST(root='./data', download=True, train=True, transform=ToTensor() )
mnist_test = datasets.MNIST(root='./data', download=True, train=False, transform=ToTensor() )

train_dataloader = DataLoader( mnist_train, batch_size=32, shuffle=True )
test_dataloader = DataLoader( mnist_test, batch_size=32, shuffle=True )

# 모델 설정
'''
입력 784: 각 이미지의 픽셀 수 (28×28=784)
출력 100: 히든 레이어의 뉴런 개수 (임의로 설정한 값)
너무 적으면: 정보 손실
너무 많으면: 과적합, 계산 비용 증가
'''
model = nn.Sequential(
    nn.Linear( 784, 100 ),
    nn.ReLU(),
    nn.Linear( 100, 1 )
)

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam( model.parameters(), lr=0.001 )

# 한 번의 반복(iteration)에서 처리되는 데이터:
for i in range( 0, 10 ):
    loss_sum = 0
    for X,y in train_dataloader:
        # print(f"X shape: {X.shape}")  # [32, 1, 28, 28] - 32개 이미지
        # print(f"y shape: {y.shape}")  # [32] - 32개 레이블
        X = X.reshape((-1, 784 )) # [32, 784] - 32개 이미지를 각각 784픽셀로
        y = (y == 0).type(torch.float32).reshape((-1,1)) # [32, 1] - 32개 결과
        
        '''
        # 배치 크기 32 예시
        원본_y = [3, 0, 7, 0, 1, 9, 0, 2, ...]  # 32개 숫자
        이진_y = [0, 1, 0, 1, 0, 0, 1, 0, ...]  # 32개 결과 (0인가?)
        최종_y = [[0], [1], [0], [1], [0], [0], [1], [0], ...]  # [32, 1] 형태
        '''

        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()

        loss_sum+=loss.item()
       
    print( loss_sum )

# 0값을 추출함 > but, 원-핫 인코딩에 대응하지 못함
model.eval()
with torch.no_grad():
    accurate = 0
    total = 0

    for X, y in test_dataloader:
        X = X.reshape((-1, 784))
        y = (y == 0).reshape((-1,1))

        outputs = nn.functional.sigmoid(model(X)) # 모델 출력을 0~1 확률로 변환
        correct_pred = ((outputs > 0.5) == y) # 0.5보다 크면 True, 작으면 False로 예측
        total+=correct_pred.size(0) # 결과: 32 (첫 번째 차원의 크기 = 배치 크기)
        accurate+=correct_pred.type(torch.int).sum().item()
        '''
        # Step 1: .type(torch.int) - 불리언을 정수로 변환
        correct_pred = [True, False, True, True, False, ...]
        정수_변환 = [1, 0, 1, 1, 0, ...]  # True→1, False→0
        
        # Step 2: .sum() - 모든 값 더하기
        합계 = 1 + 0 + 1 + 1 + 0 + ... = 3  # 맞춘 개수
        
        # Step 3: .item() - 텐서에서 Python 숫자로 추출
        최종값 = 3
        
        accurate += 최종값  # accurate = accurate + 3
        '''

    print( accurate / total )







(tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000

SystemExit: 